# Experiment: DJF 850 hPa Wind ERA5 Monthly Analysis

This notebook focuses on four diagnostics only for the ERA5 monthly 850 hPa wind data:

- **Vector climatology:** Period 1, Period 2, and delta
- **Vector anomaly:** Period 1, Period 2, and delta anomaly
- **Zonal (`u850`) climatology:** Period 1, Period 2, and delta
- **Zonal (`u850`) anomaly:** Period 1, Period 2, and delta anomaly

Regression products and related diagnostics have been removed from this working copy.


## Notebook roadmap

1. Setup and configuration
2. Helper functions
3. Wind loading and preprocessing diagnostics
4. DJF construction and seasonal diagnostics
5. Vector climatology analysis and figure export
6. Vector anomaly analysis and figure export
7. Zonal (`u850`) climatology analysis and figure export
8. Zonal (`u850`) anomaly analysis and figure export


In [ ]:
from __future__ import annotations

from pathlib import Path

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LATITUDE_FORMATTER, LONGITUDE_FORMATTER
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import Markdown, display
from scipy import stats

plt.style.use('default')
xr.set_options(keep_attrs=True)
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 140)


In [ ]:
# -----------------------------
# Config: edit these first
# -----------------------------

def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / '.git').exists():
            return candidate
    return start

REPO_ROOT = find_repo_root(Path.cwd())
NOTEBOOK_DIR = REPO_ROOT / 'notebooks' / 'circulation850hpa'
CLIMATE_DATA_DIR = REPO_ROOT / 'external' / 'ClimateData'

ERA5_PATH = CLIMATE_DATA_DIR / 'era5-monthly' / 'era5monthly_uvq_1980-2020.nc'

UWIND_VAR = 'u'
VWIND_VAR = 'v'
TIME_VAR = 'valid_time'
PRESSURE_LEVEL_HPA = 850.0
LEVEL_CANDIDATES = ('pressure_level', 'level', 'lev', 'plev')
LAT_CANDIDATES = ('latitude', 'lat', 'y')
LON_CANDIDATES = ('longitude', 'lon', 'x')

YEAR_START = 1981
YEAR_END = 2020
PERIOD1 = (1981, 2006)
PERIOD2 = (2007, 2020)

LAT_RANGE = (-21.0, 21.0)
LON_RANGE_360 = (70.0, 290.0)
CENTRAL_LONGITUDE = 180

OUTPUT_DIR = NOTEBOOK_DIR / 'output' / 'wind850_era5monthly'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

QUIVER_STRIDE = 25
QUIVER_WIDTH = 0.0022
QUIVER_SCALE_CLIM = 250
QUIVER_SCALE_ANOM = 50
QUIVER_KEY_SPEED = 5.0
QUIVER_KEY_ANOM = 1.0

USE_MAGNITUDE_SHADING = True
CLIM_SPEED_CMAP = 'YlGnBu'
ANOM_SPEED_CMAP = 'PuBuGn'
FIG_DPI = 180

print(f'Repo root: {REPO_ROOT}')
print(f'Notebook dir: {NOTEBOOK_DIR}')
print(f'Output dir: {OUTPUT_DIR}')
print(f'ERA5 path exists: {ERA5_PATH.exists()} -> {ERA5_PATH}')


## Helper functions

These helpers keep the analysis explicit while avoiding repeated code. They cover:

- longitude standardization and domain subsetting
- monthly coverage checks
- DJF-year construction
- wind magnitude calculations
- quiver-panel plotting support


In [ ]:
def resolve_name(names, candidates):
    lookup = {str(name).lower(): str(name) for name in names}
    for candidate in candidates:
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]
    return None


def standardize_lat_lon_names(data: xr.DataArray | xr.Dataset, lat_candidates=LAT_CANDIDATES, lon_candidates=LON_CANDIDATES):
    lat_name = resolve_name(data.dims, lat_candidates) or resolve_name(data.coords, lat_candidates)
    lon_name = resolve_name(data.dims, lon_candidates) or resolve_name(data.coords, lon_candidates)
    rename_map = {}
    if lat_name and lat_name != 'lat':
        rename_map[lat_name] = 'lat'
    if lon_name and lon_name != 'lon':
        rename_map[lon_name] = 'lon'
    if rename_map:
        data = data.rename(rename_map)
    if 'lat' not in data.coords or 'lon' not in data.coords:
        raise KeyError(f'Could not standardize latitude/longitude coordinates from {tuple(data.coords)}')
    return data


def ensure_lon_360(data: xr.DataArray | xr.Dataset, lon_name: str = 'lon'):
    if lon_name not in data.coords:
        raise KeyError(f'Missing longitude coordinate: {lon_name}')
    lon_values = np.mod(np.asarray(data[lon_name].values, dtype=float), 360.0)
    return data.assign_coords({lon_name: lon_values}).sortby(lon_name)


def subset_tropical_pacific_domain(data: xr.DataArray | xr.Dataset, lon_bounds=LON_RANGE_360, lat_bounds=LAT_RANGE):
    lon_min, lon_max = lon_bounds
    lat_min, lat_max = lat_bounds
    data = ensure_lon_360(data)
    data = data.sel(lon=slice(lon_min, lon_max))
    lat_values = np.asarray(data['lat'].values, dtype=float)
    if lat_values[0] < lat_values[-1]:
        data = data.sel(lat=slice(lat_min, lat_max))
    else:
        data = data.sel(lat=slice(lat_max, lat_min))
    return data


def select_pressure_level(data: xr.DataArray, target_hpa: float = PRESSURE_LEVEL_HPA, level_candidates=LEVEL_CANDIDATES) -> xr.DataArray:
    level_name = resolve_name(data.dims, level_candidates) or resolve_name(data.coords, level_candidates)
    if level_name is None:
        raise KeyError(f'Could not find a pressure-level dimension in {data.dims}')
    level_values = np.asarray(data[level_name].values, dtype=float)
    level_index = int(np.argmin(np.abs(level_values - target_hpa)))
    selected_level = float(level_values[level_index])
    if not np.isclose(selected_level, target_hpa, atol=0.5):
        raise ValueError(f'Closest level is {selected_level}, not {target_hpa}')
    print(f'Selected pressure level: {selected_level:.1f} hPa from coordinate {level_name!r}')
    return data.sel({level_name: selected_level})


def check_monthly_coverage(time_index: pd.DatetimeIndex, label: str) -> pd.DatetimeIndex:
    monthly = pd.date_range(time_index.min(), time_index.max(), freq='MS')
    missing = monthly.difference(time_index)
    print(f'{label}: first month = {time_index.min().strftime("%Y-%m")}, last month = {time_index.max().strftime("%Y-%m")}, n_months = {len(time_index)}')
    if len(missing) == 0:
        print(f'{label}: no missing monthly timestamps detected.')
    else:
        print(f'WARNING: {label} is missing {len(missing)} monthly timestamps.')
        preview = ', '.join(ts.strftime('%Y-%m') for ts in missing[:12])
        print(f'First missing months: {preview}')
    return missing


def _season_membership_table(time_index: pd.DatetimeIndex) -> pd.DataFrame:
    mask = time_index.month.isin([12, 1, 2])
    season_times = time_index[mask]
    djf_year = np.where(season_times.month == 12, season_times.year + 1, season_times.year)
    return pd.DataFrame({'time': season_times, 'month': season_times.month, 'djf_year': djf_year})


def build_complete_djf_means(monthly: xr.DataArray, start_year: int, end_year: int, label: str) -> xr.DataArray:
    time_index = pd.DatetimeIndex(monthly['time'].values)
    if pd.Timestamp(f'{start_year - 1}-12-01') not in time_index:
        print(f'WARNING: {label} is missing Dec {start_year - 1}, so DJF {start_year} cannot be built unless you accept an incomplete season.')

    membership = _season_membership_table(time_index)
    monthly = monthly.sel(time=membership['time'].to_numpy())
    djf_year_da = xr.DataArray(membership['djf_year'].to_numpy(), coords={'time': monthly['time']}, dims=('time',), name='djf_year')

    counts = membership.groupby('djf_year')['month'].nunique().sort_index()
    complete_years = counts[counts == 3].index.to_numpy(dtype=int)
    target_years = pd.Index(np.arange(start_year, end_year + 1), name='year')

    seasonal = monthly.groupby(djf_year_da).mean('time', skipna=False).rename({'djf_year': 'year'})
    seasonal = seasonal.assign_coords(year=seasonal['year'].astype(int)).sortby('year')
    seasonal = seasonal.sel(year=np.intersect1d(seasonal['year'].values.astype(int), target_years.to_numpy(dtype=int)))
    seasonal = seasonal.sel(year=np.intersect1d(seasonal['year'].values.astype(int), complete_years))

    missing_years = sorted(set(target_years.to_numpy(dtype=int)) - set(seasonal['year'].values.astype(int).tolist()))
    if missing_years:
        print(f'WARNING: {label} missing complete DJF seasons for years: {missing_years}')
    return seasonal


def period_years(period):
    return np.arange(period[0], period[1] + 1, dtype=int)


def wind_speed(u: xr.DataArray, v: xr.DataArray) -> xr.DataArray:
    return xr.apply_ufunc(np.hypot, u, v)


## Load and preprocess monthly 850 hPa wind


In [ ]:
era5_ds = xr.open_dataset(ERA5_PATH)

print('ERA5 dataset summary:')
display(era5_ds)

u_monthly = era5_ds[UWIND_VAR]
v_monthly = era5_ds[VWIND_VAR]

if TIME_VAR in u_monthly.dims:
    u_monthly = u_monthly.rename({TIME_VAR: 'time'})
    v_monthly = v_monthly.rename({TIME_VAR: 'time'})
    print(f"Renamed {TIME_VAR!r} to 'time'")

u_monthly = standardize_lat_lon_names(u_monthly)
v_monthly = standardize_lat_lon_names(v_monthly)

u850 = select_pressure_level(u_monthly, PRESSURE_LEVEL_HPA)
v850 = select_pressure_level(v_monthly, PRESSURE_LEVEL_HPA)

u850 = subset_tropical_pacific_domain(u850)
v850 = subset_tropical_pacific_domain(v850)

print('\nPost-subset diagnostics:')
print('u850 dims:', u850.dims, 'shape:', u850.shape)
print('v850 dims:', v850.dims, 'shape:', v850.shape)
print('Latitude range:', float(u850['lat'].min()), 'to', float(u850['lat'].max()))
print('Longitude range:', float(u850['lon'].min()), 'to', float(u850['lon'].max()))
print('Time coverage:', pd.DatetimeIndex(u850['time'].values).min(), 'to', pd.DatetimeIndex(u850['time'].values).max())

u_missing = check_monthly_coverage(pd.DatetimeIndex(u850['time'].values), 'u850 monthly')
v_missing = check_monthly_coverage(pd.DatetimeIndex(v850['time'].values), 'v850 monthly')

if not pd.DatetimeIndex(u850['time'].values).equals(pd.DatetimeIndex(v850['time'].values)):
    print('WARNING: u850 and v850 monthly timestamps are not identical. They will be aligned by intersection.')
    common_time = np.intersect1d(u850['time'].values, v850['time'].values)
    u850 = u850.sel(time=common_time)
    v850 = v850.sel(time=common_time)
    print(f'Aligned monthly wind timestamps to common set: {len(common_time)} months')


## Construct DJF seasonal means


In [ ]:
u850_djf = build_complete_djf_means(u850, YEAR_START, YEAR_END, label='u850')
v850_djf = build_complete_djf_means(v850, YEAR_START, YEAR_END, label='v850')

common_djf_years = np.intersect1d(u850_djf['year'].values.astype(int), v850_djf['year'].values.astype(int))
u850_djf = u850_djf.sel(year=common_djf_years)
v850_djf = v850_djf.sel(year=common_djf_years)

print('Available DJF years after preprocessing:')
print(common_djf_years.tolist())
print(f'Number of DJF seasons available for wind: {len(common_djf_years)}')

print('\nDJF construction sanity check:')
print('DJF 1981 should mean Dec 1980 + Jan 1981 + Feb 1981')
print('First available DJF year in wind data:', int(common_djf_years.min()))
print('Last available DJF year in wind data:', int(common_djf_years.max()))


## Vector climatology analysis


In [ ]:
p1_years = period_years(PERIOD1)
p2_years = period_years(PERIOD2)

print('Requested period lengths:')
print(f'Period 1 = {PERIOD1[0]}-{PERIOD1[1]} ({len(p1_years)} DJF seasons requested)')
print(f'Period 2 = {PERIOD2[0]}-{PERIOD2[1]} ({len(p2_years)} DJF seasons requested)')

print('\nAvailable wind seasons in each period:')
print('Period 1 years:', np.intersect1d(common_djf_years, p1_years).tolist())
print('Period 2 years:', np.intersect1d(common_djf_years, p2_years).tolist())

u_clim_full = u850_djf.mean('year')
v_clim_full = v850_djf.mean('year')

u_clim_p1 = u850_djf.sel(year=np.intersect1d(common_djf_years, p1_years)).mean('year')
v_clim_p1 = v850_djf.sel(year=np.intersect1d(common_djf_years, p1_years)).mean('year')

u_clim_p2 = u850_djf.sel(year=np.intersect1d(common_djf_years, p2_years)).mean('year')
v_clim_p2 = v850_djf.sel(year=np.intersect1d(common_djf_years, p2_years)).mean('year')

u_clim_delta = u_clim_p2 - u_clim_p1
v_clim_delta = v_clim_p2 - v_clim_p1

climatology_panels = [
    {'title': f'DJF climatology\n{PERIOD1[0]}-{PERIOD1[1]}', 'u': u_clim_p1, 'v': v_clim_p1, 'mag': wind_speed(u_clim_p1, v_clim_p1)},
    {'title': f'DJF climatology\n{PERIOD2[0]}-{PERIOD2[1]}', 'u': u_clim_p2, 'v': v_clim_p2, 'mag': wind_speed(u_clim_p2, v_clim_p2)},
    {'title': f'Delta climatology\n({PERIOD2[0]}-{PERIOD2[1]}) - ({PERIOD1[0]}-{PERIOD1[1]})', 'u': u_clim_delta, 'v': v_clim_delta, 'mag': wind_speed(u_clim_delta, v_clim_delta)},
]

climatology_png = OUTPUT_DIR / 'wind850_djf_climatology_3panel_p1_p2_delta.png'
proj = ccrs.PlateCarree(central_longitude=CENTRAL_LONGITUDE)
data_crs = ccrs.PlateCarree()
fig, axes = plt.subplots(
    nrows=3,
    ncols=1,
    figsize=(10, 6),
    subplot_kw={'projection': proj},
    constrained_layout=True,
)

max_mag = max(float(np.nanmax(panel['mag'].values)) for panel in climatology_panels)
vmax = max_mag if max_mag > 0 else 1.0

shading_mesh = None
quiver_artist = None
for idx, (ax, panel) in enumerate(zip(axes, climatology_panels)):
    shading_mesh = ax.pcolormesh(
        panel['u']['lon'],
        panel['u']['lat'],
        panel['mag'],
        cmap=CLIM_SPEED_CMAP,
        vmin=0.0,
        vmax=vmax,
        shading='auto',
        transform=data_crs,
    )

    u_thin = panel['u'].isel(lat=slice(None, None, QUIVER_STRIDE), lon=slice(None, None, QUIVER_STRIDE))
    v_thin = panel['v'].isel(lat=slice(None, None, QUIVER_STRIDE), lon=slice(None, None, QUIVER_STRIDE))
    quiver_artist = ax.quiver(
        u_thin['lon'].values,
        u_thin['lat'].values,
        u_thin.values,
        v_thin.values,
        transform=data_crs,
        scale=QUIVER_SCALE_CLIM,
        width=QUIVER_WIDTH,
        color='black',
        pivot='mid',
    )

    ax.set_extent([LON_RANGE_360[0], LON_RANGE_360[1], LAT_RANGE[0], LAT_RANGE[1]], crs=data_crs)
    ax.coastlines(linewidth=0.7)
    ax.add_feature(cfeature.BORDERS, linewidth=0.35)
    gl = ax.gridlines(
        crs=data_crs,
        draw_labels=True,
        xlocs=np.arange(90, 301, 30),
        ylocs=np.arange(-15, 16, 5),
        linewidth=0.4,
        color='0.45',
        alpha=0.4,
        linestyle='--',
    )
    gl.top_labels = False
    gl.right_labels = False
    gl.left_labels = True
    gl.bottom_labels = idx == len(climatology_panels) - 1
    gl.xformatter = LONGITUDE_FORMATTER
    gl.yformatter = LATITUDE_FORMATTER
    gl.xlabel_style = {'size': 9}
    gl.ylabel_style = {'size': 9}

    ax.set_title(panel['title'], fontsize=11)

cbar = fig.colorbar(
    shading_mesh,
    ax=axes,
    orientation='vertical',
    fraction=0.035,
    pad=0.03,
)
cbar.set_label('Wind speed magnitude (m s$^{-1}$)')

axes[0].quiverkey(
    quiver_artist,
    0.02,
    1.05,
    QUIVER_KEY_SPEED,
    f'{QUIVER_KEY_SPEED:g} m s$^{{-1}}$',
    labelpos='E',
    coordinates='axes',
)

fig.suptitle('DJF 850 hPa wind climatology and period difference', fontsize=14)
fig.savefig(climatology_png, dpi=FIG_DPI, bbox_inches='tight')
print(f'Saved: {climatology_png}')


## Vector anomaly analysis


In [ ]:
u_anom_p1 = u_clim_p1 - u_clim_full
v_anom_p1 = v_clim_p1 - v_clim_full
u_anom_p2 = u_clim_p2 - u_clim_full
v_anom_p2 = v_clim_p2 - v_clim_full
u_anom_delta = u_anom_p2 - u_anom_p1
v_anom_delta = v_anom_p2 - v_anom_p1

anomaly_panels = [
    {'title': f'Period-mean anomaly\n{PERIOD1[0]}-{PERIOD1[1]} relative to {YEAR_START}-{YEAR_END}', 'u': u_anom_p1, 'v': v_anom_p1, 'mag': wind_speed(u_anom_p1, v_anom_p1)},
    {'title': f'Period-mean anomaly\n{PERIOD2[0]}-{PERIOD2[1]} relative to {YEAR_START}-{YEAR_END}', 'u': u_anom_p2, 'v': v_anom_p2, 'mag': wind_speed(u_anom_p2, v_anom_p2)},
    {'title': f'Delta anomaly\n({PERIOD2[0]}-{PERIOD2[1]}) anomaly - ({PERIOD1[0]}-{PERIOD1[1]}) anomaly', 'u': u_anom_delta, 'v': v_anom_delta, 'mag': wind_speed(u_anom_delta, v_anom_delta)},
]

anomaly_png = OUTPUT_DIR / 'wind850_djf_anomaly_3panel_p1_p2_delta.png'
proj = ccrs.PlateCarree(central_longitude=CENTRAL_LONGITUDE)
data_crs = ccrs.PlateCarree()
fig, axes = plt.subplots(
    nrows=3,
    ncols=1,
    figsize=(10, 6),
    subplot_kw={'projection': proj},
    constrained_layout=True,
)

max_mag = max(float(np.nanmax(panel['mag'].values)) for panel in anomaly_panels)
vmax = max_mag if max_mag > 0 else 1.0

shading_mesh = None
quiver_artist = None
for idx, (ax, panel) in enumerate(zip(axes, anomaly_panels)):
    shading_mesh = ax.pcolormesh(
        panel['u']['lon'],
        panel['u']['lat'],
        panel['mag'],
        cmap=ANOM_SPEED_CMAP,
        vmin=0.0,
        vmax=2.1,
        shading='auto',
        transform=data_crs,
    )

    u_thin = panel['u'].isel(lat=slice(None, None, QUIVER_STRIDE), lon=slice(None, None, QUIVER_STRIDE))
    v_thin = panel['v'].isel(lat=slice(None, None, QUIVER_STRIDE), lon=slice(None, None, QUIVER_STRIDE))
    quiver_artist = ax.quiver(
        u_thin['lon'].values,
        u_thin['lat'].values,
        u_thin.values,
        v_thin.values,
        transform=data_crs,
        scale=QUIVER_SCALE_ANOM,
        width=QUIVER_WIDTH,
        color='black',
        pivot='mid',
    )

    ax.set_extent([LON_RANGE_360[0], LON_RANGE_360[1], LAT_RANGE[0], LAT_RANGE[1]], crs=data_crs)
    ax.coastlines(linewidth=0.7)
    ax.add_feature(cfeature.BORDERS, linewidth=0.35)
    gl = ax.gridlines(
        crs=data_crs,
        draw_labels=True,
        xlocs=np.arange(90, 301, 30),
        ylocs=np.arange(-15, 16, 5),
        linewidth=0.4,
        color='0.45',
        alpha=0.4,
        linestyle='--',
    )
    gl.top_labels = False
    gl.right_labels = False
    gl.left_labels = True
    gl.bottom_labels = idx == len(anomaly_panels) - 1
    gl.xformatter = LONGITUDE_FORMATTER
    gl.yformatter = LATITUDE_FORMATTER
    gl.xlabel_style = {'size': 9}
    gl.ylabel_style = {'size': 9}

    ax.set_title(panel['title'], fontsize=11)

cbar = fig.colorbar(
    shading_mesh,
    ax=axes,
    orientation='vertical',
    fraction=0.035,
    pad=0.03,
)
cbar.set_label('Anomaly vector magnitude (m s$^{-1}$)')

axes[0].quiverkey(
    quiver_artist,
    0.02,
    1.05,
    QUIVER_KEY_ANOM,
    f'{QUIVER_KEY_ANOM:g} m s$^{{-1}}$',
    labelpos='E',
    coordinates='axes',
)

fig.suptitle('DJF 850 hPa wind anomalies relative to the full-period climatology', fontsize=14)
fig.savefig(anomaly_png, dpi=FIG_DPI, bbox_inches='tight')
print(f'Saved: {anomaly_png}')


## u850 component analysis

This section produces scalar maps of the zonal (`u850`) component only to support explicit interpretation of westerly versus easterly changes.


### u850 climatology maps


In [ ]:
u_clim_full = u850_djf.mean('year')
u_clim_p1 = u850_djf.sel(year=np.intersect1d(common_djf_years, p1_years)).mean('year')
u_clim_p2 = u850_djf.sel(year=np.intersect1d(common_djf_years, p2_years)).mean('year')
u_clim_delta = u_clim_p2 - u_clim_p1

# Full-period climatology individual plot
u850_clim_full_png = OUTPUT_DIR / 'u850_climatology_full_period.png'
proj = ccrs.PlateCarree(central_longitude=CENTRAL_LONGITUDE)
data_crs = ccrs.PlateCarree()
fig, ax = plt.subplots(
    nrows=1,
    ncols=1,
    figsize=(10, 8),
    subplot_kw={'projection': proj},
    constrained_layout=True,
)

full_vabs_max = np.nanmax(np.abs(u_clim_full.values))
full_vabs_max = full_vabs_max if full_vabs_max > 0 else 1.0

mesh = ax.pcolormesh(
    u_clim_full['lon'],
    u_clim_full['lat'],
    u_clim_full,
    cmap='BrBG',
    vmin=-full_vabs_max,
    vmax=full_vabs_max,
    shading='auto',
    transform=data_crs,
)

ax.set_extent([LON_RANGE_360[0], LON_RANGE_360[1], LAT_RANGE[0], LAT_RANGE[1]], crs=data_crs)
ax.coastlines(linewidth=0.7)
ax.add_feature(cfeature.BORDERS, linewidth=0.35)
gl = ax.gridlines(
    crs=data_crs,
    draw_labels=True,
    xlocs=np.arange(90, 301, 30),
    ylocs=np.arange(-15, 16, 5),
    linewidth=0.4,
    color='0.45',
    alpha=0.4,
    linestyle='--',
)
gl.xformatter = LONGITUDE_FORMATTER
gl.yformatter = LATITUDE_FORMATTER
gl.xlabel_style = {'size': 9}
gl.ylabel_style = {'size': 9}

ax.set_title(f'DJF u850 climatology (full period {YEAR_START}-{YEAR_END})', fontsize=12)

cbar = fig.colorbar(mesh, ax=ax, orientation='vertical', fraction=0.035, pad=0.03)
cbar.set_label('Zonal wind u850 (m s$^{-1}$)')

fig.suptitle('DJF 850 hPa Zonal Wind (u850) Full-Period Climatology', fontsize=14)
fig.savefig(u850_clim_full_png, dpi=FIG_DPI, bbox_inches='tight')
print(f'Saved: {u850_clim_full_png}')

# 3-panel climatology: P1, P2, Delta (with BrBG colormap)
u_clim_panels = [
    {'title': f'DJF u850 climatology\n{PERIOD1[0]}-{PERIOD1[1]}', 'data': u_clim_p1},
    {'title': f'DJF u850 climatology\n{PERIOD2[0]}-{PERIOD2[1]}', 'data': u_clim_p2},
    {'title': f'Delta u850 climatology\n({PERIOD2[0]}-{PERIOD2[1]}) - ({PERIOD1[0]}-{PERIOD1[1]})', 'data': u_clim_delta},
]

u850_clim_png = OUTPUT_DIR / 'u850_climatology_p1_p2_delta.png'
fig, axes = plt.subplots(
    nrows=3,
    ncols=1,
    figsize=(10, 6),
    subplot_kw={'projection': proj},
    constrained_layout=True,
)

# Compute color limits for P1 and P2, and use a fixed separate range for delta
p12_data = [u_clim_p1.values, u_clim_p2.values]
p12_vabs_max = max(np.nanmax(np.abs(d)) for d in p12_data)
p12_vabs_max = p12_vabs_max if p12_vabs_max > 0 else 1.0

delta_vmin, delta_vmax = -2.0, 2.0

mesh_list = []
for idx, (ax, panel) in enumerate(zip(axes, u_clim_panels)):
    if idx < 2:
        vmin, vmax = -p12_vabs_max, p12_vabs_max
        cmap = 'BrBG'
    else:
        vmin, vmax = delta_vmin, delta_vmax
        cmap = 'PiYG'
    
    mesh = ax.pcolormesh(
        panel['data']['lon'],
        panel['data']['lat'],
        panel['data'],
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        shading='auto',
        transform=data_crs,
    )
    mesh_list.append(mesh)
    
    ax.set_extent([LON_RANGE_360[0], LON_RANGE_360[1], LAT_RANGE[0], LAT_RANGE[1]], crs=data_crs)
    ax.coastlines(linewidth=0.7)
    ax.add_feature(cfeature.BORDERS, linewidth=0.35)
    gl = ax.gridlines(
        crs=data_crs,
        draw_labels=True,
        xlocs=np.arange(90, 301, 30),
        ylocs=np.arange(-15, 16, 5),
        linewidth=0.4,
        color='0.45',
        alpha=0.4,
        linestyle='--',
    )
    gl.top_labels = False
    gl.right_labels = False
    gl.left_labels = True
    gl.bottom_labels = idx == 2
    gl.xformatter = LONGITUDE_FORMATTER
    gl.yformatter = LATITUDE_FORMATTER
    gl.xlabel_style = {'size': 9}
    gl.ylabel_style = {'size': 9}
    
    ax.set_title(panel['title'], fontsize=11)

# Add colorbars
cbar1 = fig.colorbar(mesh_list[0], ax=axes[0:2], orientation='vertical', fraction=0.035, pad=0.03)
cbar1.set_label('Zonal wind u850 (m s$^{-1}$)')
cbar2 = fig.colorbar(mesh_list[2], ax=axes[2], orientation='vertical', fraction=0.035, pad=0.03)
cbar2.set_label('Δu850 (m s$^{-1}$)')

fig.suptitle('DJF 850 hPa Zonal Wind (u850): Period 1, Period 2, and Delta', fontsize=14)
fig.savefig(u850_clim_png, dpi=FIG_DPI, bbox_inches='tight')
print(f'Saved: {u850_clim_png}')

print('\nDJF u850 climatology summary:')
print(f'Full period ({YEAR_START}-{YEAR_END}): min={float(u_clim_full.min()):.3f}, max={float(u_clim_full.max()):.3f}, mean={float(u_clim_full.mean()):.3f} m/s')
print(f'Period 1 ({PERIOD1[0]}-{PERIOD1[1]}): min={float(u_clim_p1.min()):.3f}, max={float(u_clim_p1.max()):.3f}, mean={float(u_clim_p1.mean()):.3f} m/s')
print(f'Period 2 ({PERIOD2[0]}-{PERIOD2[1]}): min={float(u_clim_p2.min()):.3f}, max={float(u_clim_p2.max()):.3f}, mean={float(u_clim_p2.mean()):.3f} m/s')
print(f'Delta ({PERIOD2[0]}-{PERIOD2[1]} minus {PERIOD1[0]}-{PERIOD1[1]}): min={float(u_clim_delta.min()):.3f}, max={float(u_clim_delta.max()):.3f}, mean={float(u_clim_delta.mean()):.3f} m/s')

### u850 anomaly maps


In [ ]:
u_anom_p1 = u_clim_p1 - u_clim_full
u_anom_p2 = u_clim_p2 - u_clim_full
u_anom_delta = u_anom_p2 - u_anom_p1

u_anom_panels = [
    {'title': f'u850 anomaly\n{PERIOD1[0]}-{PERIOD1[1]} (rel. to {YEAR_START}-{YEAR_END})', 'data': u_anom_p1},
    {'title': f'u850 anomaly\n{PERIOD2[0]}-{PERIOD2[1]} (rel. to {YEAR_START}-{YEAR_END})', 'data': u_anom_p2},
    {'title': f'Delta u850 anomaly\n({PERIOD2[0]}-{PERIOD2[1]}) - ({PERIOD1[0]}-{PERIOD1[1]}) anomalies', 'data': u_anom_delta},
]

u850_anom_png = OUTPUT_DIR / 'u850_anomaly_maps.png'
proj = ccrs.PlateCarree(central_longitude=CENTRAL_LONGITUDE)
data_crs = ccrs.PlateCarree()
fig, axes = plt.subplots(
    nrows=3,
    ncols=1,
    figsize=(10, 6),
    subplot_kw={'projection': proj},
    constrained_layout=True,
)

anom_vabs_max = max(np.nanmax(np.abs(panel['data'].values)) for panel in u_anom_panels)
anom_vabs_max = anom_vabs_max if anom_vabs_max > 0 else 0.5

mesh = None
for idx, (ax, panel) in enumerate(zip(axes, u_anom_panels)):
    mesh = ax.pcolormesh(
        panel['data']['lon'],
        panel['data']['lat'],
        panel['data'],
        cmap='RdBu_r',
        vmin=-anom_vabs_max,
        vmax=anom_vabs_max,
        shading='auto',
        transform=data_crs,
    )
    
    ax.set_extent([LON_RANGE_360[0], LON_RANGE_360[1], LAT_RANGE[0], LAT_RANGE[1]], crs=data_crs)
    ax.coastlines(linewidth=0.7)
    ax.add_feature(cfeature.BORDERS, linewidth=0.35)
    gl = ax.gridlines(
        crs=data_crs,
        draw_labels=True,
        xlocs=np.arange(90, 301, 30),
        ylocs=np.arange(-15, 16, 5),
        linewidth=0.4,
        color='0.45',
        alpha=0.4,
        linestyle='--',
    )
    gl.top_labels = False
    gl.right_labels = False
    gl.left_labels = idx == 0
    gl.bottom_labels = True
    gl.xformatter = LONGITUDE_FORMATTER
    gl.yformatter = LATITUDE_FORMATTER
    gl.xlabel_style = {'size': 9}
    gl.ylabel_style = {'size': 9}
    
    ax.set_title(panel['title'], fontsize=11)

cbar = fig.colorbar(mesh, ax=axes, orientation='vertical', fraction=0.025, pad=0.03)
cbar.set_label('u850 anomaly (m s$^{-1}$)')

fig.suptitle('DJF 850 hPa Zonal Wind Anomalies (relative to full-period climatology)', fontsize=14)
fig.savefig(u850_anom_png, dpi=FIG_DPI, bbox_inches='tight')
print(f'Saved: {u850_anom_png}')

print('\nDJF u850 anomaly summary:')
print(f'Period 1 anomaly: min={float(u_anom_p1.min()):.3f}, max={float(u_anom_p1.max()):.3f}, mean={float(u_anom_p1.mean()):.3f} m/s')
print(f'Period 2 anomaly: min={float(u_anom_p2.min()):.3f}, max={float(u_anom_p2.max()):.3f}, mean={float(u_anom_p2.mean()):.3f} m/s')
print(f'Delta anomaly: min={float(u_anom_delta.min()):.3f}, max={float(u_anom_delta.max()):.3f}, mean={float(u_anom_delta.mean()):.3f} m/s')